# Step 15 — End-to-End Demo Recording
**Tough Talks · Phase 6**

Goal: bring up the full Tough Talks stack on Colab — Gemma 4 multimodal model, FastAPI backend, vanilla-JS frontend, and a public tunnel — so the demo can be driven from a local browser and screen-recorded. Unlike Step 14 (which drove every route in-process via `TestClient`), this step launches a real long-running uvicorn server and exposes it to the public internet via a cloudflared quick-tunnel.

**Architecture (set up by Step 14, unchanged here):**

```
Colab GPU runtime
└── uvicorn :8000  (single process — multimodal Gemma 4 loaded by the lifespan)
    ├── /app/        → frontend/app.html (+ app.js, app.css)
    ├── /storage/*   → local JSON at data/local/
    ├── /talk-dna /vault /persona /premortem /debrief /aftermath /pulse
    ├── /transcribe /emotion
    └── /health

       │  cloudflared quick-tunnel (no signup, ephemeral URL)
       ▼
   Local browser → https://<random>.trycloudflare.com/app/
```

**What `done` looks like for this step:**

1. Uvicorn boots with `TOUGH_TALKS_INCLUDE_TEXT=0` — multimodal-only per the constrained-VRAM rule (`knowledge/phases/rules.md`). Text routes are served by `ModelRegistry.text()`'s fallback to the multimodal pair.
2. `/health` reports `multimodal_model_loaded=true` and `frontend_present=true`.
3. cloudflared opens a public quick-tunnel and the notebook captures its `https://*.trycloudflare.com` URL.
4. The tunnel URL serves both `/app/` (frontend) and every API route from the same origin — verified by an out-of-process round-trip through the tunnel.
5. Final cell prints the click-through URL plus an unconditional pass/fail table (same shape as Steps 12 / 13 / 14).

**Demo script (drive from the browser once the URL is live):**

Settings (leave Base URL empty) → Person Vault: build *Jamie* → Talk DNA: analyse → Pre-Mortem → Practice Round: 3 turns → Debrief → Aftermath → Save → second round → Insights → Pulse → (optional) Live Mode: upload a short clip for `/transcribe` + `/emotion`.

In [1]:
# ── 0. Install dependencies + cloudflared ────────────────────────────
# Same install line-up as Step 14 plus `uvicorn[standard]` for the
# stand-alone ASGI server (Step 14 used `TestClient` in-process and
# didn't need it). cloudflared ships as a single static binary —
# permissive-licensed (per `[[project_kaggle_license_safety]]`) and
# the quick-tunnel mode requires no account.
#
# After this first run, RESTART THE KERNEL before continuing if you
# actually upgraded transformers.

!pip install -q -U transformers accelerate
!pip install -q fastapi 'pydantic>=2.6' 'uvicorn[standard]' python-multipart httpx soundfile librosa gTTS requests

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 135.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
cloudflared version 2026.5.0 (built 2026-05-13-11:24 UTC)


In [2]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────────
# Same shim as every prior step.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"],
                   capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"],
                   capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()),
                pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed:\n" + result.stderr)
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_stale = [m for m in list(sys.modules) if m == "backend" or m.startswith("backend.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"Cleared {len(_stale)} cached backend.* module(s) from sys.modules")

print(f"Repo root: {REPO_ROOT}")

Cloning https://github.com/EhsanFarazmand/tough_talks.git -> /content/tough_talks
Repo root: /content/tough_talks


In [3]:
# ── 2. Imports ──────────────────────────────────────────────
import json
import re
import signal
import subprocess
import threading
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

import requests

In [4]:
# ── 3. Config ─────────────────────────────────────────────────
HOST = "127.0.0.1"
PORT = 8000
BOOT_TIMEOUT_S = 300   # multimodal Gemma 4 cold-load on T4 fits inside 5 min
TUNNEL_TIMEOUT_S = 60  # cloudflared usually surfaces a URL within ~10 s

STORAGE_ROOT = Path(REPO_ROOT) / "data" / "local"
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
UVICORN_LOG = Path("/tmp/tough_talks_uvicorn.log")
TUNNEL_LOG = Path("/tmp/tough_talks_cloudflared.log")

# Constrained-VRAM rule (knowledge/phases/rules.md § Decoding knobs by
# task type, and the Step 12 single-model rule): T4 only fits the
# multimodal variant. Text routes fall back via ModelRegistry.text()'s
# multimodal-pair shim. Setting these env vars BEFORE Popen so the
# uvicorn subprocess inherits them and the lifespan loader honours
# them on the very first request.
os.environ["TOUGH_TALKS_INCLUDE_TEXT"] = "0"
os.environ["TOUGH_TALKS_INCLUDE_MULTIMODAL"] = "1"
os.environ["TOUGH_TALKS_STORAGE_ROOT"] = str(STORAGE_ROOT)
# TOUGH_TALKS_FRONTEND_DIR left unset — resolve_frontend_dir() defaults
# to <repo>/frontend, which is exactly where Step 14 ships the bundle.

print("Host          :", f"{HOST}:{PORT}")
print("Storage root  :", STORAGE_ROOT)
print("uvicorn log   :", UVICORN_LOG)
print("tunnel log    :", TUNNEL_LOG)
print("Env knobs     :", {k: v for k, v in os.environ.items() if k.startswith("TOUGH_TALKS_")})

Host          : 127.0.0.1:8000
Storage root  : /content/tough_talks/data/local
uvicorn log   : /tmp/tough_talks_uvicorn.log
tunnel log    : /tmp/tough_talks_cloudflared.log
Env knobs     : {'TOUGH_TALKS_INCLUDE_TEXT': '0', 'TOUGH_TALKS_INCLUDE_MULTIMODAL': '1', 'TOUGH_TALKS_STORAGE_ROOT': '/content/tough_talks/data/local'}


In [5]:
# ── 4. Launch uvicorn in the background ──────────────────────────────
# The FastAPI lifespan loads the multimodal Gemma 4 variant on the
# first boot — ~60-120 s on a clean T4 (longer if HF Hub is throttling
# unauthenticated downloads). Logs stream to UVICORN_LOG so the next
# cell can tail them on failure without blocking the kernel.

_uvicorn_log_f = open(UVICORN_LOG, "w")
uvicorn_proc = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn", "backend.api.main:app",
        "--host", HOST, "--port", str(PORT),
        "--log-level", "info",
    ],
    stdout=_uvicorn_log_f,
    stderr=subprocess.STDOUT,
    cwd=str(REPO_ROOT),
)
print(f"uvicorn started (pid={uvicorn_proc.pid})")
print(f"  log: {UVICORN_LOG}")
print("  first boot loads the multimodal Gemma 4 variant — give it ~60–120 s")

uvicorn started (pid=3858)
  log: /tmp/tough_talks_uvicorn.log
  first boot loads the multimodal Gemma 4 variant — give it ~60–120 s


In [6]:
# ── 5. Wait for /health to report the model loaded ──────────────────────
HEALTH_URL = f"http://{HOST}:{PORT}/health"
HEALTH_BODY: dict | None = None
deadline = time.time() + BOOT_TIMEOUT_S

while time.time() < deadline:
    if uvicorn_proc.poll() is not None:
        print(f"uvicorn exited unexpectedly (returncode={uvicorn_proc.returncode}). Tail of log:")
        print(UVICORN_LOG.read_text()[-3000:])
        raise RuntimeError("uvicorn died during boot")
    try:
        with urlopen(HEALTH_URL, timeout=3) as resp:
            body = json.loads(resp.read())
            if body.get("multimodal_model_loaded"):
                HEALTH_BODY = body
                break
            print(f"  /health reachable, model loading… (mm_loaded={body.get('multimodal_model_loaded')})")
    except (URLError, ConnectionRefusedError, TimeoutError, ConnectionResetError):
        pass
    time.sleep(3)

if HEALTH_BODY is None:
    print("Health did not turn green within %d s. Tail of uvicorn log:" % BOOT_TIMEOUT_S)
    print(UVICORN_LOG.read_text()[-3000:])
    raise RuntimeError("uvicorn /health never reported multimodal_model_loaded=true")

print("READY")
print(json.dumps(HEALTH_BODY, indent=2))

READY
{
  "status": "ok",
  "version": "0.1.0",
  "text_model_loaded": false,
  "multimodal_model_loaded": true,
  "storage_root": "/content/tough_talks/data/local",
  "frontend_dir": "/content/tough_talks/frontend",
  "frontend_present": true
}


In [9]:
# ── 6. Open a cloudflared quick-tunnel ────────────────────────────────
# Quick-tunnels expose a random https://*.trycloudflare.com URL. No
# account, no auth, no DNS setup. The URL rotates every restart —
# re-run THIS cell only if the tunnel dies (uvicorn keeps running).
#
# Same-origin frontend (Step 14) means one tunnel covers both the UI
# and every API route — the browser fetches `/storage/...` etc.
# directly off the trycloudflare URL with no CORS gymnastics.

TUNNEL_URL: str | None = None
TUNNEL_RE = re.compile(r"https://[a-z0-9\-]+\.trycloudflare\.com")
_tunnel_log_f = open(TUNNEL_LOG, "w")

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://{HOST}:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

deadline = time.time() + TUNNEL_TIMEOUT_S
while time.time() < deadline and TUNNEL_URL is None:
    if tunnel_proc.poll() is not None:
        print("cloudflared exited unexpectedly. Tail of tunnel log:")
        print(TUNNEL_LOG.read_text()[-2000:])
        raise RuntimeError("cloudflared died before printing a URL")
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue
    _tunnel_log_f.write(line)
    _tunnel_log_f.flush()
    m = TUNNEL_RE.search(line)
    if m:
        TUNNEL_URL = m.group(0)
        break

def _drain_tunnel_log() -> None:
    """Background-drain cloudflared's stdout so its pipe buffer never fills."""
    for line in tunnel_proc.stdout:
        _tunnel_log_f.write(line)
        _tunnel_log_f.flush()

threading.Thread(target=_drain_tunnel_log, daemon=True).start()

if TUNNEL_URL is None:
    print("Tunnel URL not captured within %d s. Tail of tunnel log:" % TUNNEL_TIMEOUT_S)
    print(TUNNEL_LOG.read_text()[-2000:])
    raise RuntimeError("cloudflared did not surface a quick-tunnel URL")

print("Tunnel URL  :", TUNNEL_URL)
print("Frontend at :", TUNNEL_URL + "/app/")
print("API docs at :", TUNNEL_URL + "/docs")

Tunnel URL  : https://compiler-set-wiring-printing.trycloudflare.com
Frontend at : https://compiler-set-wiring-printing.trycloudflare.com/app/
API docs at : https://compiler-set-wiring-printing.trycloudflare.com/docs


In [8]:
# ── 6. Open an ngrok tunnel (no 100 s edge timeout) ───────────────────
!pip install -q pyngrok
from pyngrok import ngrok, conf

NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"   # ngrok.com → Your Authtoken
ngrok.set_auth_token(NGROK_TOKEN)
conf.get_default().region = "us"        # pick the region closest to you

# Kill any leftover ngrok session from a previous cell
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public = ngrok.connect(PORT, "http")
TUNNEL_URL = public.public_url.replace("http://", "https://")
print("Tunnel URL  :", TUNNEL_URL)
print("Frontend at :", TUNNEL_URL + "/app/")
print("API docs at :", TUNNEL_URL + "/docs")

# Stub the tunnel_proc so cells 7 / 8 / teardown still work
class _StubProc:
    def __init__(self): self.pid = -1
    def poll(self): return None
    def terminate(self): pass
tunnel_proc = _StubProc()

ERROR:pyngrok.process.ngrok:t=2026-05-16T09:53:16+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: PASTE_YOUR_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-16T09:53:16+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: PASTE_YOUR_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-16T09:53:16+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: The authtoken you specified does not look like a proper ngrok aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok authtoken.\nYour authtoken: PASTE_YOUR_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

In [ ]:
# ── 7. End-to-end validation through the public tunnel ────────────────────
# Proves the entire request path — browser -> trycloudflare edge ->
# cloudflared on Colab -> uvicorn -> FastAPI -> static mount / route
# handler — actually works before we hand the URL to the screen-
# recording software. Out-of-process requests via `requests` (not
# TestClient — that was Step 14's surface).

RESULTS: list[dict] = []

def _record(name: str, ok: bool, detail: str) -> None:
    RESULTS.append({"check": name, "ok": ok, "detail": detail})
    print(f"{'PASS' if ok else 'FAIL'}  {name:42s} {detail}")

_record("uvicorn alive", uvicorn_proc.poll() is None, f"pid={uvicorn_proc.pid}")
_record("cloudflared alive", tunnel_proc.poll() is None, f"pid={tunnel_proc.pid}")
_record(
    "multimodal model loaded",
    bool(HEALTH_BODY.get("multimodal_model_loaded")),
    f"text={HEALTH_BODY.get('text_model_loaded')}, mm={HEALTH_BODY.get('multimodal_model_loaded')}",
)
_record(
    "frontend dir present",
    bool(HEALTH_BODY.get("frontend_present")),
    str(HEALTH_BODY.get("frontend_dir")),
)
_record(
    "storage root present",
    bool(HEALTH_BODY.get("storage_root")),
    str(HEALTH_BODY.get("storage_root")),
)

try:
    r = requests.get(f"{TUNNEL_URL}/health", timeout=20)
    r.raise_for_status()
    body = r.json()
    _record("GET tunnel /health", body.get("status") == "ok", f"status={body.get('status')}")
except Exception as exc:  # noqa: BLE001
    _record("GET tunnel /health", False, f"{type(exc).__name__}: {exc}")

try:
    r = requests.get(f"{TUNNEL_URL}/app/", timeout=20)
    r.raise_for_status()
    _record("GET tunnel /app/", "<title>Tough Talks" in r.text, f"len={len(r.text)} bytes")
except Exception as exc:  # noqa: BLE001
    _record("GET tunnel /app/", False, f"{type(exc).__name__}: {exc}")

for asset, sentinel in (
    ("/app/app.js", "const api = {"),
    ("/app/app.css", "--gold:"),
):
    try:
        r = requests.get(f"{TUNNEL_URL}{asset}", timeout=20)
        r.raise_for_status()
        _record(f"GET tunnel {asset}", sentinel in r.text, f"len={len(r.text)} bytes")
    except Exception as exc:  # noqa: BLE001
        _record(f"GET tunnel {asset}", False, f"{type(exc).__name__}: {exc}")

try:
    r = requests.get(f"{TUNNEL_URL}/storage/vault", timeout=20)
    r.raise_for_status()
    body = r.json()
    _record(
        "GET tunnel /storage/vault",
        "items" in body and "count" in body,
        f"count={body.get('count')}",
    )
except Exception as exc:  # noqa: BLE001
    _record("GET tunnel /storage/vault", False, f"{type(exc).__name__}: {exc}")

## Demo flow — drive from your local browser

1. Open the **Tunnel URL** printed above with `/app/` appended.
2. **Settings** card — leave the *Base URL* field empty (the app is same-origin).
3. **Person Vault** — name `Jamie`, relationship `colleague`, paste a 4-turn transcript, **Build profile**, then **Save**.
4. **Talk DNA** — paste the same transcript, **Analyse**, then **Save**.
5. **Pre-Mortem** — describe the conversation + goal, **Generate**.
6. **Practice Round** — start a round and run 3 user turns; the persona replies each time.
7. **Debrief + Aftermath** — at the end of the round, click **Debrief**, then **Aftermath**, then **Save bundle**.
8. Run a second round so Pulse has ≥2 rounds to roll up.
9. **Insights** — **Refresh Pulse**.
10. (Optional) **Live Mode** — record a short clip via the browser mic, upload for `/transcribe` + `/emotion/analyze`. Keep clips under 30 s (Gemma 4 audio cap); longer clips are auto-windowed by `transcribe_long`.

### Mid-recording debug

If something looks off while recording, peek at the log files from a fresh cell without restarting anything:

```python
!tail -n 40 /tmp/tough_talks_uvicorn.log
!tail -n 40 /tmp/tough_talks_cloudflared.log
```

### Persistence

All artefacts saved during the demo land under `STORAGE_ROOT` (`<repo>/data/local/`). To survive a Colab disconnect, copy them off before the runtime expires:

```python
!cp -r data/local /content/drive/MyDrive/tough_talks_demo_data  # after mounting Drive
```

In [ ]:
# ── 8. Final pass/fail summary (renders unconditionally) ──────────────────
print()
print("=" * 78)
print("TOUGH TALKS — STEP 15 LIVE DEMO")
print("=" * 78)
print()
print(f"  Open in your browser  :  {TUNNEL_URL}/app/")
print(f"  API docs              :  {TUNNEL_URL}/docs")
print(f"  Health endpoint       :  {TUNNEL_URL}/health")
print(f"  Storage root (Colab)  :  {STORAGE_ROOT}")
print()

if not RESULTS:
    print("No results recorded — the validation cell did not run.")
else:
    name_w = max(len(r["check"]) for r in RESULTS)
    header = f"  {'check':{name_w}s}  status  detail"
    rule = "  " + "-" * (name_w + 8) + "-" * 60
    print(header)
    print(rule)
    for row in RESULTS:
        mark = "PASS" if row["ok"] else "FAIL"
        print(f"  {row['check']:{name_w}s}  {mark:6s}  {row['detail']}")
    print(rule)
    passed = sum(1 for r in RESULTS if r["ok"])
    total = len(RESULTS)
    print(f"  {passed}/{total} checks OK")

## Teardown

Run the cell below between recording takes if you need to free the port and re-open a fresh tunnel URL. The model weights stay loaded in VRAM only as long as the uvicorn process is alive, so terminating it forces a full reload on the next boot — only do this if a take is actually broken.

In [14]:
# ── 9. Teardown (uncomment to use) ────────────────────────────────────
tunnel_proc.terminate()
try: tunnel_proc.wait(timeout=5)
except subprocess.TimeoutExpired: tunnel_proc.kill()

uvicorn_proc.send_signal(signal.SIGINT)  # trigger lifespan shutdown
try: uvicorn_proc.wait(timeout=15)
except subprocess.TimeoutExpired: uvicorn_proc.kill()

_tunnel_log_f.close(); _uvicorn_log_f.close()
print("uvicorn + cloudflared stopped")

NameError: name 'tunnel_proc' is not defined